# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR^2 dataset using the `mlcroissant` Python library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using mlcroissant. This step fetches the schema and initializes the Dataset object, which you will use for further exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant JSON-LD schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Initialize the Dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as object attributes
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")
print(f"Version: {dataset.metadata.version}")
print(f"Authors: {dataset.metadata.author}")

## 2. Data Overview

Let's examine the available record sets and their structure. All record sets, fields, and columns are referenced by their `@id` (as required for Croissant datasets).

In [ ]:
# List all record sets and their field IDs
print('Available RecordSets (referenced by @id):')
for rs in dataset.metadata.recordSet:
    print(f"  RecordSet @id: {rs['@id']}")
    # List fields for the record set
    if 'field' in rs:
        fields = rs['field']
        print('    Fields:')
        for fld in fields:
            # Each field is a dict or a reference
            if isinstance(fld, dict):
                field_id = fld.get('@id', str(fld))
            else:
                field_id = str(fld)
            print(f"      - {field_id}")
    if 'column' in rs:
        columns = rs['column']
        print('    Columns:')
        for col in columns:
            if isinstance(col, dict):
                col_id = col.get('@id', str(col))
            else:
                col_id = str(col)
            print(f"      - {col_id}")
print('\n---\n')

# List the first few records from all record sets to get a preview
for rs in dataset.metadata.recordSet:
    rs_id = rs['@id']
    print(f"Example record from {rs_id}:")
    for i, rec in enumerate(dataset.records(record_set=rs_id)):
        print(rec)
        if i > 0:
            break
    print()

## 3. Data Extraction

Now, we'll load all records from each record set into a pandas DataFrame. 
**Note:** You should use the `@id` string for each record set.

We'll list all record set IDs below:

In [ ]:
# Extract all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSet]
print('Record set @ids:', record_set_ids)

# Load each record set into a DataFrame, indexed by @id
dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records for {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(f"  Sample data:\n{df.head()}\n")
    except Exception as e:
        print(f"  Could not load records: {e}\n")

# For further analysis, we'll use the first available record set
if record_set_ids:
    chosen_rs_id = record_set_ids[0]
    print(f"Chosen record set for analysis: {chosen_rs_id}")
else:
    chosen_rs_id = None

## 4. Exploratory Data Analysis (EDA)

Let's analyze numerical and categorical data in the dataset. We'll choose a numeric and a group/categorical field for standard preprocessing tasks.

- Outlier filtering
- Normalization
- Grouping

Edit variables below if you want to examine a particular field.

In [ ]:
import numpy as np

# Pick a numeric field and a group field by their @id as seen above
# Below are example fallback values; edit as appropriate for this dataset:
if chosen_rs_id and not dataframes[chosen_rs_id].empty:
    df = dataframes[chosen_rs_id]
    # Guess a numeric field
    numeric_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.int64] or pd.api.types.is_numeric_dtype(df[col])]
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
    else:
        # Try to parse some as numeric
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
                if pd.api.types.is_numeric_dtype(df[col]):
                    numeric_field = col
                    break
            except:
                continue
        else:
            numeric_field = None
    print(f"Using numeric field: {numeric_field}")

    # Group/categorical field (excluding index and pure numeric fields)
    group_candidates = [col for col in df.columns if df[col].dtype == object]
    group_field = None
    if group_candidates:
        group_field = group_candidates[0]
    print(f"Using group/categorical field: {group_field}")

    # Proceed only if numeric field is valid
    if numeric_field:
        # Remove outliers (e.g., keep within 2 stddevs)
        mean = df[numeric_field].mean()
        std = df[numeric_field].std()
        lower, upper = mean - 2*std, mean + 2*std
        filtered_df = df[(df[numeric_field] >= lower) & (df[numeric_field] <= upper)]
        print(f"Filtered {numeric_field} for outliers (within 2 std): {filtered_df.shape[0]} records remain.")

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean) / std
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group and aggregate
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Mean {numeric_field} grouped by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric field available for EDA.")
else:
    print('No suitable DataFrame for EDA. Check previous cells for data loading issues.')

## 5. Visualization

Let's visualize the distribution of the numeric field, and (if possible) the group-wise means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting numeric field distribution
if chosen_rs_id and not dataframes[chosen_rs_id].empty and numeric_field:
    plt.figure(figsize=(7, 4))
    sns.histplot(data=filtered_df, x=numeric_field, kde=True)
    plt.title(f"Distribution of {numeric_field} (outliers removed)")
    plt.xlabel(numeric_field)
    plt.show()

    # Grouped barplot if group_field is present
    if group_field and 'grouped_df' in locals():
        plt.figure(figsize=(10,4))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f"Mean {numeric_field} per {group_field}")
        plt.xticks(rotation=30)
        plt.show()
else:
    print('No suitable data to visualize. Ensure previous steps succeeded.')

## 6. Conclusion

- Explored and loaded the FAIR^2 dataset with `mlcroissant`, referencing all entities by their `@id`.
- Inspected schema and records, loaded them into DataFrames, and applied exploratory data analysis and visualization.
- The fields and record sets may vary by update/version; always check the actual field names and types using the code above before deeper analysis.

This template can be adapted to other Croissant-format datasets to ensure reproducible FAIR exploration and downstream ML use.